In [1]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
import datetime as dt
import gsw
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import ArcTools as Atools

import matplotlib.path as mpath
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import ArcTools as Atools

In [3]:
# read
dir_clim = '/data0/user/aprigent/ISAS/ISAS17_Mask.nc'
mask = xr.open_dataset(dir_clim)
zi = mask.depthCFD.data

# Vertical interpolation function


In [4]:
def interp_prof(x_old, y_old, x_new):

    # Import interpolation methods from SciPy
    # (only Akima1DInterpolator is actually used below)
    from scipy.interpolate import Akima1DInterpolator
    
    yi = []   # list to store interpolated profiles
    iyi = []  # list to store indices of successfully interpolated profiles
    
    # Loop over each profile (rows of y_old)
    for i in range(0, y_old.shape[0]):
        
        # Extract the i-th profile
        y = y_old[i, :]
        x = x_old[i, :]
        
        # Identify non-NaN values in y and x_old
        inans_y = ~np.isnan(y)
        inans_x = ~np.isnan(x)
        
        # Keep only points where both x and y are valid
        inans = inans_y * inans_x
        
        # Proceed only if there are any valid points
        if np.any(inans):
            
            # Need at least two valid points for interpolation
            if len(y[inans]) > 1:

                # Create Akima spline interpolator using valid data points
                spl = Akima1DInterpolator(x[inans], y[inans])
                
                # Interpolate profile onto the new x grid
                prof = spl(x_new)
                
                # Store interpolated profile and its original index
                yi.append(prof)
                iyi.append(i)
                
    # Return:
    # - array of interpolated profiles
    # - array of indices of profiles that were successfully interpolated
    return np.array(yi), np.array(iyi)


In [5]:
## NOT USED ##

def extrap_prof(param):
    """ Extrapolates the first valid value upward
    """
    
    # Loop over each profile (assumes param is 2D: profiles x depth)
    for j in range(0, len(param)): 
        
        # Extract the j-th profile
        prof = param[j, :]
        
        # Start from the first depth level
        i = 0
        x = prof[i]
        
        # Move downward in the profile while:
        #  - the current value is NaN
        #  - OR we are still within the first 4 indices (i <= 3)
        while np.isnan(x) | i <= 3:
            x = prof[i]   # get current value
            i = i + 1     # move to next depth level
        
        # Fill all levels above index i with the found value x
        # → effectively extrapolates the first valid value upward
        prof[:i] = x
    
    # Return the modified array (already changed in place)
    return param


In [6]:
itp_path = '/data0/user/aprigent/ITP/'
list_file = glob.glob(itp_path + '*DM_final.nc')
list_file

['/data0/user/aprigent/ITP/ITP_13_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_72_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_131_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_63_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_47_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_49_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_132_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_102_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_71_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_122_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_42_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_36_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_56_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_75_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_109_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_37_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_23_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_16_L3_DM_final.nc',
 '/data0/user/aprigent/ITP/ITP_92_L3_DM_final.nc',
 '/data0/user/aprigent/ITP

In [7]:
test = xr.open_dataset('/data0/user/aprigent/ITP/ITP_9_L3_DM_final.nc',decode_times=False)
test

<xarray.Dataset>
Dimensions:              (N_PROF: 1480, N_LEVELS: 786)
Dimensions without coordinates: N_PROF, N_LEVELS
Data variables: (12/63)
    REFERENCE_DATE_TIME  |S16 ...
    DATA_TYPE            |S16 ...
    PLATFORM_NUMBER      (N_PROF) |S8 ...
    WMO_INST_TYPE        (N_PROF) |S4 ...
    PI_NAME              (N_PROF) |S64 ...
    DATA_CENTRE          (N_PROF) |S4 ...
    ...                   ...
    PRES_ADJUSTED        (N_PROF, N_LEVELS) float32 ...
    PRES_ADJUSTED_CLMN   (N_PROF, N_LEVELS) float32 ...
    PRES_ADJUSTED_CLSD   (N_PROF, N_LEVELS) float32 ...
    PRES_ADJUSTED_ERME   (N_PROF, N_LEVELS) float32 ...
    PRES_ADJUSTED_ERROR  (N_PROF, N_LEVELS) float32 ...
    PRES_ADJUSTED_RESI   (N_PROF, N_LEVELS) float32 ...
Attributes:
    Conventions:       CF-1.4
    title:             ITP DATA
    institution:       LOPS/IUEM
    project_name:      SNO-ARGO/ISAS
    data_manager:      Nicolas Kolodziejczyk
    software_version:  v2
    references:        The Ice-Tethered Profiler data were collected and made...
    history:           20260318T095739L : Creation

In [8]:
test.TEMP_ADJUSTED_ERROR.min()

<xarray.DataArray 'TEMP_ADJUSTED_ERROR' ()>
array(0., dtype=float32)

# Start looping on ITP individual files

In [9]:
black_list = [itp_path+'ITP_132_L3_DM_final.nc',\
              itp_path+'ITP_66_L3_DM_final.nc',\
             itp_path+'ITP_55_L3_DM_final.nc',\
             itp_path+'ITP_71_L3_DM_final.nc',\
             itp_path+'ITP_8_L3_DM_final.nc']

data_out = '/data0/user/aprigent/ITP/'
for f in list_file:
    itp = xr.open_dataset(f,decode_times=False)
    juld_qc = itp.JULD_QC.data
    qc_time = juld_qc == b'1'
    # extract ITP number
    itp_id = int(os.path.basename(f).split('_')[1])
    print(itp_id)
    juld_ord = [dt.datetime(1950,1,1).toordinal() + t for t in itp.JULD.data[qc_time]]
    if len(juld_ord) == 0:
        print('No QC time')
        continue
        
    if juld_ord[-1]> dt.datetime(1999,12,31).toordinal():
        if f not in black_list:
            # print(f)
            pres = itp.PRES_ADJUSTED.data[qc_time]
            lat = itp.LATITUDE.data[qc_time]
            lon = itp.LONGITUDE.data[qc_time]
            psal= itp.PSAL_ADJUSTED.data[qc_time]
            temp=itp.TEMP_ADJUSTED.data[qc_time]
            psal_qc = itp.PSAL_ADJUSTED_QC.data[qc_time]
            temp_qc = itp.TEMP_ADJUSTED_QC.data[qc_time]
            pos_qc = itp.POSITION_QC.data[qc_time]
            pres_qc =itp.PRES_ADJUSTED_QC.data[qc_time] 
            data_mode = itp.DATA_MODE.data[qc_time]
            
            qc_press = pres_qc == b'1'
            pres[~qc_press] = np.nan
            dep = -gsw.z_from_p(pres.T,lat).T
            
            qc1_psal = psal_qc == b'1'
            qc1_temp = temp_qc == b'1'
            qc1_pos = pos_qc == b'1'
            qc_mode = (data_mode == b'D')


            
            qc_mode_2d_psal = np.broadcast_to(qc_mode[:, None], psal.shape)
            qc_mode_2d_temp = np.broadcast_to(qc_mode[:, None], temp.shape)
            qc_pos_2d_psal = np.broadcast_to(qc1_pos[:, None], psal.shape)
            qc_pos_2d_temp = np.broadcast_to(qc1_pos[:, None], temp.shape)

            final_mask_temp = ~(qc_pos_2d_temp & qc1_temp & qc_press & qc_mode_2d_temp)
            final_mask_psal = ~(qc_pos_2d_psal & qc1_psal & qc_press & qc_mode_2d_psal)


            psal[final_mask_psal] = np.nan
            temp[final_mask_temp] = np.nan  
            
            
            n_levels_temp = np.ones((dep.shape[0]))*np.sum(np.isfinite(dep[:,:]),axis=1)
            n_levels_psal  = np.ones((dep.shape[0]))*np.sum(np.isfinite(dep[:,:]),axis=1)

            sum_temp = np.nansum(temp,axis=1)
            sum_psal  = np.nansum(psal,axis=1)
            sum_dep = np.nansum(dep,axis=1)
            
    
            PSAL,ipsal = interp_prof(dep,psal,zi)
            TEMP,itemp = interp_prof(dep,temp,zi)
            
            
            
            
            if len(ipsal) != len(itemp):
                print('psal/temp dimension missmatch')
            TIMEs = np.array(juld_ord)[ipsal]
            LONs = lon[ipsal]
            LATs = lat[ipsal]
            sum_PSAL = sum_psal[ipsal]
            n_levels_PSAL = n_levels_psal[ipsal]
            sum_dep_PSAL = sum_dep[ipsal]
            
            
            TIMEt = np.array(juld_ord)[itemp]
            LONt = lon[itemp]
            LATt = lat[itemp]
            sum_TEMP = sum_temp[itemp]
            n_levels_TEMP = n_levels_temp[itemp]
            sum_dep_TEMP = sum_dep[itemp]
            
            TEMP[TEMP<-2] = np.nan
            TEMP[TEMP>30] = np.nan
            PSAL[PSAL<0] = np.nan
            PSAL[PSAL>40] = np.nan
            
            itp_id_PSAL = np.full(len(TIMEs), itp_id)
            itp_id_TEMP = np.full(len(TIMEt), itp_id)
            
            
            print('TEMP max = ',np.nanmax(TEMP))
            print('TEMP min = ',np.nanmin(TEMP))
            print('PSAL max = ',np.nanmax(PSAL))
            print('PSAL min = ',np.nanmin(PSAL))
            
            
            # --- Daily averaging before saving ---
            # TIMEs_int, LONs_int, LATs_int, PSAL_int, COUNTs = daily_average(TIMEs, LONs, LATs, PSAL)
            # TIMEt_int, LONt_int, LATt_int, TEMP_int, COUNTt = daily_average(TIMEt, LONt, LATt, TEMP)
            print(f[25:])
        
            # save file PSAL
            ds = xr.Dataset({'latitude': (['prof'], LATs),
                             'longitude': (['prof'], LONs),
                             'time': (['prof'], TIMEs),
                             'salinity': (['prof','levels'], PSAL),
                             'sum_salinity': (['prof'], sum_PSAL),
                             'sum_levels': (['prof'], sum_dep_PSAL),
                             'nb_levels': (['prof'], n_levels_PSAL),
                             'prof_descr': (['prof'],np.char.add('ITP', itp_id_PSAL.astype(str)).astype('S30')),
                             'depth': (['levels'], zi)},
                            coords={'prof': (['prof'], np.arange(0,len(TIMEs))), 'levels':(['levels'], zi)})
            
            # global attributes
            ds.attrs['Comments'] = 'ITP salinity profiles (qc=1) interpolated on ISAS vertical levels using a Akima 1D interpolator scheme (scipy)'
            ds.attrs['Original file'] = f[25:]
            ds.attrs['title'] = 'ITP salinity profiles on ISAS vertical grid'
            ds.attrs['source'] = 'ITP'
            ds.attrs['history'] = 'Created with xarray'
            ds['prof_descr'].attrs = {'long_name': 'Ice-Tethered Profiler ID'}
            
            # variable attributes
            ds['latitude'].attrs = {'long_name':'Latitude','units':'degrees_north','standard_name':'latitude'}
            ds['longitude'].attrs = {'long_name':'Longitude','units':'degrees_east','standard_name':'longitude'}
            ds['time'].attrs = {'long_name':'Time',
                                "units": "days since 0001-01-01 (Python datetime system)"}
            ds['salinity'].attrs = {'long_name':'Salinity (S78 - PSS)',
                                    'units':'1e-3',   # or 'psu'
                                    'standard_name':'sea_water_practical_salinity'}
            ds['depth'].attrs = {'long_name':'Depth','units':'m','positive':'down','standard_name':'depth'}
            


            ds.to_netcdf(data_out + f[25:-3] + '_ISAS_PSAL.nc')   
            
            # save file TEMP
            ds = xr.Dataset({'latitude': (['prof'], LATt),\
                             'longitude': (['prof'], LONt),\
                             'time': (['prof'], TIMEt),\
                             'temperature': (['prof','levels'], TEMP),\
                             'sum_temperature': (['prof'], sum_TEMP),
                             'sum_levels': (['prof'], sum_dep_TEMP),
                             'nb_levels': (['prof'], n_levels_TEMP),
                             'prof_descr': (['prof'],np.char.add('ITP', itp_id_TEMP.astype(str)).astype('S30')),
                             'depth': (['levels'], zi)},\
                            coords={'prof': (['prof'], np.arange(0,len(TIMEt))), 'levels':(['levels'], zi)})

            # global attributes
            ds.attrs['Comments'] = 'ITP temperature profiles (qc=1) interpolated on ISAS vertical using a Akima 1D interpolator scheme (scipy)'
            ds.attrs['Original file'] = f[25:]
            ds.attrs['title'] = 'ITP temperature profiles on ISAS vertical grid'
            ds.attrs['source'] = 'ITP'
            ds.attrs['history'] = 'Created with xarray'
            ds['prof_descr'].attrs = {'long_name': 'Ice-Tethered Profiler ID'}
            
            # variable attributes
            ds['latitude'].attrs = {'long_name':'Latitude','units':'degrees_north','standard_name':'latitude'}
            ds['longitude'].attrs = {'long_name':'Longitude','units':'degrees_east','standard_name':'longitude'}
            ds['time'].attrs = {'long_name':'Time',
                                "units": "days since 0001-01-01 (Python datetime system)"}
            ds['temperature'].attrs = {'long_name':'Temperature (T90)',
                                       'units':'degree_Celsius',
                                       'standard_name':'sea_water_temperature'}
            ds['depth'].attrs = {'long_name':'Depth','units':'m','positive':'down','standard_name':'depth'}

  
            ds.to_netcdf(data_out + f[25:-3] + '_ISAS_TEMP.nc')         

13
TEMP max =  1.1082187675785315
TEMP min =  -1.6574256814272488
PSAL max =  34.886050884330395
PSAL min =  23.62588561068064
ITP_13_L3_DM_final.nc
72
TEMP max =  1.58523357973919
TEMP min =  -1.8305412569724675
PSAL max =  34.9237582942463
PSAL min =  30.60773677299164
ITP_72_L3_DM_final.nc
131
TEMP max =  0.9662488443604668
TEMP min =  -1.5548451906967031
PSAL max =  34.8752051741939
PSAL min =  26.70334684197121
ITP_131_L3_DM_final.nc
63
psal/temp dimension missmatch
TEMP max =  0.6329725230324839
TEMP min =  -1.6861179960181807
PSAL max =  34.8916120344254
PSAL min =  29.05322545167336
ITP_63_L3_DM_final.nc
47
psal/temp dimension missmatch
TEMP max =  1.859091566946638
TEMP min =  -1.8142596008991692
PSAL max =  34.89949171080213
PSAL min =  30.82176517203518
ITP_47_L3_DM_final.nc
49
TEMP max =  1.333848933575242
TEMP min =  -1.7591320167040165
PSAL max =  34.89015964600733
PSAL min =  28.16043721598719
ITP_49_L3_DM_final.nc
132
102
TEMP max =  1.6188649040074683
TEMP min =  -1.83